# Método dos Gradientes Conjugados — Versão Limpa e Didática

Este notebook apresenta uma versão refatorada, limpa e comentada do algoritmo desenvolvido no meu **TCC**.

### O Problema
Queremos encontrar o ponto $x^* \in \mathbb{R}^n$ que minimiza a função quadrática convexa:
$$f(x) = \frac{1}{2} x^T A x + b^T x + c$$
onde $A$ é uma matriz **simétrica e definida positiva** ($n \times n$). No ponto de mínimo, o gradiente $\nabla f(x) = Ax + b$ é igual ao vetor nulo ($0$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Funções Base: Função Objetivo e Gradiente
Em vez de usar `np.dot` aninhados ou variáveis globais, usamos o operador `@` do Python (multiplicação matricial/vetorial) e passamos `x`, `A` e `b` como parâmetros claros:

In [ ]:
def funcao_quadratica(x: np.ndarray, A: np.ndarray, b: np.ndarray, c: float = 0.0) -> float:
    """Calcula o valor escalar f(x) = (1/2) * x^T A x + b^T x + c."""
    return 0.5 * (x @ A @ x) + (b @ x) + c


def gradiente(x: np.ndarray, A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Calcula o vetor gradiente ∇f(x) = A x + b."""
    return A @ x + b

## 2. O Algoritmo dos Gradientes Conjugados

A ideia central do método é escolher direções de busca $d_k$ que sejam **$A$-ortogonais (conjugadas)** entre si ($d_i^T A d_j = 0$). Isso garante que, em uma função quadrática de dimensão $n$, o algoritmo encontre o mínimo exato em **no máximo $n$ passos**.

### O que mudou em relação ao código original?
1. **Critério de parada matemático (`np.linalg.norm(g) < tol`):** Em vez da função `comparar()` que comparava elemento por elemento com `0` e retornava texto (`"igual"` / `"Não igual"`), medimos o tamanho (norma euclidiana) do vetor gradiente.
2. **Sem repetição de cálculos:** Calculamos `Ad = A @ d` e `curvatura = d @ Ad` uma única vez por iteração.
3. **Histórico de trajetória:** Guardamos os pontos visitados para poder visualizar o caminho até o mínimo.

In [ ]:
def gradiente_conjugado(A: np.ndarray, b: np.ndarray, x0: np.ndarray, tol: float = 1e-8, max_iter: int = 100):
    x = x0.astype(float).copy()
    historico = [x.copy()]

    # Passo 0: Gradiente inicial e primeira direção (máxima descida)
    g = gradiente(x, A, b)
    d = -g

    for k in range(max_iter):
        norma_g = np.linalg.norm(g)
        print(f"Iteração {k:2d} | x = {np.round(x, 6)} | ||grad|| = {norma_g:.2e}")

        # Se o gradiente já é praticamente zero, chegamos ao ponto de mínimo
        if norma_g < tol:
            print(f"\n✅ Ponto de mínimo encontrado em {k} iterações!")
            break

        # Produto auxiliar reutilizado no passo t e no coeficiente beta
        Ad = A @ d
        curvatura = d @ Ad  # d^T A d

        # 1. Tamanho ótimo do passo (t_k)
        t = -(g @ d) / curvatura

        # 2. Novo ponto (x_{k+1})
        x = x + t * d
        historico.append(x.copy())

        # 3. Novo gradiente (g_{k+1})
        g_novo = gradiente(x, A, b)

        # 4. Coeficiente de conjugação (Beta_k)
        beta = (Ad @ g_novo) / curvatura

        # 5. Nova direção conjugada (d_{k+1})
        d = -g_novo + beta * d
        g = g_novo

    return x, np.array(historico)

## 3. Testando com os Dados Originais do TCC
No caso em que $A = 15I$ (múltiplo da matriz identidade), as curvas de nível são círculos perfeitos e a primeira direção de busca já aponta diretamente para o mínimo (converge em **1 passo**):

In [ ]:
A_tcc = np.array([[15.0, 0.0], [0.0, 15.0]])
b_tcc = np.array([8.0, 3.0])
x0_tcc = np.array([10.0, 10.0])

x_min_tcc, hist_tcc = gradiente_conjugado(A_tcc, b_tcc, x0_tcc)
print("Ponto mínimo:", x_min_tcc)

## 4. Visualizando o Poder do Método em uma Matriz Elíptica ($n = 2$ passos)
Quando $A$ possui termos fora da diagonal, as curvas de nível são elipses inclinadas. Enquanto o método do Gradiente Descendente comum faria um zigue-zague longo, o **Gradiente Conjugado** chega ao mínimo exato em apenas **2 passos**:

In [ ]:
A_elipse = np.array([[6.0, 2.0], [2.0, 3.0]])
b_elipse = np.array([-4.0, 2.0])
x0_elipse = np.array([8.0, 8.0])

x_min, trajetoria = gradiente_conjugado(A_elipse, b_elipse, x0_elipse)

# Plotando as curvas de nível e a trajetória
x1_vals = np.linspace(-3, 10, 200)
x2_vals = np.linspace(-5, 10, 200)
X1, X2 = np.meshgrid(x1_vals, x2_vals)
Z = 0.5 * (A_elipse[0, 0]*X1**2 + 2*A_elipse[0, 1]*X1*X2 + A_elipse[1, 1]*X2**2) + b_elipse[0]*X1 + b_elipse[1]*X2

plt.figure(figsize=(8, 6))
plt.contour(X1, X2, Z, levels=25, cmap="viridis", alpha=0.7)
plt.plot(trajetoria[:, 0], trajetoria[:, 1], "ro-", linewidth=2, markersize=7, label="Trajetória Gradiente Conjugado")
plt.plot(x_min[0], x_min[1], "y*", markersize=15, markeredgecolor="black", label="Ponto de Mínimo $x^*$")
plt.title("Convergência em 2 Passos do Método dos Gradientes Conjugados")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()